# Genre Classifier
Trains **Logistic Regression**, **SVM**, **Random Forest**, **Naïve Bayes**, **KNN**, and **XGBoost** models to classify book genre from
100-word text partitions.

**Inputs required (run the other notebooks first):**
- `partitions.csv` — from `book_partitioner.ipynb`
- `vocabulary.csv` — from `book_ngram_analysis.ipynb`

**Outputs:** accuracy scores, classification reports, and confusion matrices.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
import joblib

## Configuration

In [ ]:
PARTITIONS_CSV = "partitions.csv"   # output of book_partitioner.ipynb
VOCABULARY_CSV = "vocabulary.csv"   # output of book_ngram_analysis.ipynb

TEST_SIZE      = 0.2    # 80/20 train-test split
RANDOM_STATE   = 42
CV_FOLDS       = 5      # stratified k-fold cross-validation

# Genres in a consistent display order
GENRES = ["autobiography", "fantasy", "horror", "romance", "sci-fi"]


## Load Data

In [ ]:
partitions = pd.read_csv(PARTITIONS_CSV)
vocabulary  = pd.read_csv(VOCABULARY_CSV)["word"].tolist()

print(f"Partitions loaded : {len(partitions):,} rows")
print(f"Vocabulary size   : {len(vocabulary):,} words")
print(f"\nGenre distribution:")
print(partitions['genre'].value_counts().to_string())
partitions.head(3)


## Testing All Models with All 3 Vectorizers
Methods include BOW, TF-IDF, and TF-IDF N-grams

In [ ]:
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

X_text = partitions["text"].tolist()
y      = partitions["genre"].tolist()

# Define three vectorizers
methods = {
    "Bag of Words": CountVectorizer(vocabulary=vocabulary, token_pattern=r'[a-z]+'),
    "TF-IDF": TfidfVectorizer(vocabulary=vocabulary, sublinear_tf=True, token_pattern=r'[a-z]+'),
    "TF-IDF N-grams": TfidfVectorizer(vocabulary=vocabulary, sublinear_tf=True, token_pattern=r'[a-z]+', ngram_range=(1, 2))
}

# For testing removal of vocabulary restriction:
""" methods = {
    "Bag of Words": CountVectorizer(token_pattern=r'[a-z]+', min_df=3, max_df=0.7),
    "TF-IDF": TfidfVectorizer(sublinear_tf=True, token_pattern=r'[a-z]+', min_df=3, max_df=0.7),
    "TF-IDF N-grams": TfidfVectorizer(sublinear_tf=True, token_pattern=r'[a-z]+', ngram_range=(1, 2), min_df=3, max_df=0.7)
} """

models_to_test = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        C=1.0,
        solver='lbfgs',
        multi_class='multinomial',
    ),
    "SVM": LinearSVC(
        max_iter=2000,
        random_state=RANDOM_STATE,
        C=1.0,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Multinomial Naïve Bayes": MultinomialNB(
        alpha=1.0
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5, 
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        tree_method='hist',
        n_jobs=-1,
        verbosity=0,
    ),
}

for method_name, vectorizer in methods.items():
    print(f"\n=== Vectorizer: {method_name} ===")
    X = vectorizer.fit_transform(X_text)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    for model_name, model in models_to_test.items():
        model_clone = clone(model)
        
        if model_name == "XGBoost":
            le = LabelEncoder()
            y_train_encoded = le.fit_transform(y_train)
            model_clone.fit(X_train, y_train_encoded)
            y_pred = le.inverse_transform(model_clone.predict(X_test))
        else:
            model_clone.fit(X_train, y_train)
            y_pred = model_clone.predict(X_test)

        print(f"\n>> {model_name}")
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(classification_report(y_test, y_pred, target_names=GENRES))

## 10-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# Pre-encode labels once
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Setup Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# Store results 
results = []

for method_name, vectorizer in methods.items():
    print(f"\nProcessing Vectorizer: {method_name}...")
    X = vectorizer.fit_transform(X_text)
    
    for model_name, model in models_to_test.items():
        # Get 10 scores
        scores = cross_val_score(model, X, y_encoded, cv=cv, scoring='accuracy')
        
        # Save results to a list for analysis
        results.append({
            "Vectorizer": method_name,
            "Model": model_name,
            "Mean_Accuracy": np.mean(scores),
            "Std_Dev": np.std(scores)
        })
        
        print(f"  {model_name}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Convert to DataFrame to find champion
results_df = pd.DataFrame(results)
print("\n--- Final Summary ---")
print(results_df.sort_values(by="Mean_Accuracy", ascending=False).to_string(index=False))

## Feature Importance for All Models

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone

# Encode labels 
le = LabelEncoder()
y_encoded = le.fit_transform(y)

def audit_all_models(methods, models, X_data, y_encoded_data):
    for method_name, vectorizer in methods.items():
        print(f"\n{'='*20}\nAUDIT: {method_name}\n{'='*20}")
        
        # Transform data
        X = vectorizer.fit_transform(X_data)
        feature_names = vectorizer.get_feature_names_out()
        
        for model_name, model in models.items():
            print(f"\n--- Model: {model_name} ---")
            
            # Create a fresh, empty copy
            model_clone = clone(model)
            
            # Fit using the encoded labels
            model_clone.fit(X, y_encoded_data)
            
            # Extract weights based on model type
            if hasattr(model_clone, 'coef_'):
                # For Linear models (Logistic/SVM), coef_ is usually shape (n_classes, n_features)
                # If binary, it's (1, n_features). If multiclass, it's (n_classes, n_features)
                if model_clone.coef_.shape[0] == 1:
                    weights = model_clone.coef_[0]
                else:
                    # Just take the first class weights for the audit
                    weights = model_clone.coef_[0]
            elif hasattr(model_clone, 'feature_importances_'):
                # For Tree-based models
                weights = model_clone.feature_importances_
            else:
                print("Skipping: Model type does not support feature extraction.")
                continue
            
            # Zip, Sort, and Print
            feats = sorted(zip(weights, feature_names))
            
            print(f"Top 5 Indicative Words:")
            for w, f in feats[-5:]:
                print(f"  {f}: {w:.4f}")

# RUN IT using the new y_encoded
audit_all_models(methods, models_to_test, X_text, y_encoded)